# Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()))

from utils import (
    data_processing,
    get_freqs,
    compute_hs,
    compute_bulk_params,
    find_significant_peaks,
    find_peak_windows,
    classify_partition,
    classify_partitions,
)
from utils.spectral_partitioning import jonswap_spectrum, pm_at_peak


# Configuration

The only parameter that needs editing to explore a different buoy. See `buoy_data/` for
available IDs (each must contain `density.txt`, `alpha1.txt`, `alpha2.txt`, `r1.txt`, `r2.txt`,
`wind.txt`).

In [ ]:
BUOY_ID = "32012"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
FOLDER_PATH = PROJECT_ROOT / "buoy_data" / BUOY_ID
SAVE_PATH = FOLDER_PATH / "processed_data.pkl"


# Dataset Loading

In [ ]:
if SAVE_PATH.exists():
    density, alpha_1, alpha_2, r_1, r_2, wind = pd.read_pickle(SAVE_PATH)
    print(f"Loaded cached preprocessed data for buoy {BUOY_ID}")
else:
    density, alpha_1, alpha_2, r_1, r_2, wind = data_processing(
        str(FOLDER_PATH), save_path=str(SAVE_PATH)
    )

freqs = get_freqs(density).numpy()
print(f"{len(density)} hourly records, {len(freqs)} frequency bins "
      f"({freqs.min():.4f}-{freqs.max():.4f} Hz)")


# Data Overview

Sanity-check the record before computing statistics on it: date range, coverage, and how many
timestamps are effectively flat-calm (near-zero total energy) — these get skipped by peak
detection later on and would otherwise silently dilute modality/wind-sea-vs-swell percentages.

In [ ]:
print("Date range:", density.index.min(), "->", density.index.max())
print("Duration:", density.index.max() - density.index.min())
print()
print(density.describe().T[["mean", "std", "min", "max"]].head())


In [ ]:
m0_total = np.trapezoid(density.values, freqs, axis=1)
calm_frac = (m0_total <= 1e-6).mean()
print(f"Flat-calm timestamps (m0 <= 1e-6): {calm_frac:.2%}")

VALID_MASK = m0_total > 1e-6


# Bulk Wave Parameters

`Hs = 4*sqrt(m0)` and `Tm02 = sqrt(m0/m2)` via `utils.compute_bulk_params`, plus wind speed and
direction ("coming from" convention, matching `utils.data_processing.process_wind`) derived from
`wind_u`/`wind_v`.

In [ ]:
hs, tm02 = compute_bulk_params(density.values, freqs)
hs = pd.Series(hs, index=density.index, name="Hs")
tm02 = pd.Series(tm02, index=density.index, name="Tm02")

wind_speed = np.hypot(wind["wind_u"], wind["wind_v"])
wind_dir = np.degrees(np.arctan2(-wind["wind_u"], -wind["wind_v"])) % 360

wind_coverage = wind_speed.notna().mean()
print(f"Wind data coverage: {wind_coverage:.1%}")
if wind_coverage < 0.5:
    print("Low/no wind coverage for this buoy (common for a wave-only buoy with no "
          "anemometer) -- wind-dependent plots below will be sparse or empty.")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(hs.index, hs.values, lw=0.6, alpha=0.6, label="Hs (hourly)")
axes[0].plot(hs.index, hs.rolling(24, center=True).mean(), lw=1.5, label="Hs (24h rolling mean)")
axes[0].set_ylabel("Hs [m]")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(wind_speed.index, wind_speed.values, lw=0.6, color="tab:orange", label="Wind speed")
axes[1].set_ylabel("Wind speed [m/s]")
axes[1].set_xlabel("Time")
axes[1].legend()
axes[1].grid(True)

plt.suptitle(f"Buoy {BUOY_ID}: significant wave height and wind speed")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(hs.dropna(), bins=50, color="tab:blue")
axes[0].set_xlabel("Hs [m]")
axes[0].set_ylabel("Count")
axes[0].set_title("Hs distribution")
axes[0].grid(True)

axes[1].hist(tm02.dropna(), bins=50, color="tab:green")
axes[1].set_xlabel("Tm02 [s]")
axes[1].set_title("Tm02 distribution")
axes[1].grid(True)

plt.tight_layout()
plt.show()


# Spectral Shape

A few individual spectra at low/median/high Hs, plus the mean spectrum and inter-percentile band
across the whole record. The frequency grid is log-spaced (dense near 0.02 Hz, coarse near
0.485 Hz), so a log-x axis is used throughout.

In [ ]:
def plot_spectrum(ax, spectrum, freqs, label=None, **kwargs):
    ax.plot(freqs, spectrum, label=label, **kwargs)
    ax.set_xscale("log")
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel("Spectral density [m$^2$/Hz]")
    ax.grid(True, which="both", alpha=0.3)


In [ ]:
valid_hs = hs[VALID_MASK]
low_t, med_t, high_t = valid_hs.idxmin(), valid_hs.index[len(valid_hs) // 2], valid_hs.idxmax()

fig, ax = plt.subplots(figsize=(9, 5))
for t, label in [(low_t, "low Hs"), (med_t, "median Hs"), (high_t, "high Hs")]:
    plot_spectrum(ax, density.loc[t].values, freqs, label=f"{label} ({t.date()}, Hs={hs[t]:.2f}m)")
ax.legend()
ax.set_title(f"Buoy {BUOY_ID}: example spectra")
plt.tight_layout()
plt.show()


In [ ]:
density_valid = density.values[VALID_MASK]
mean_spectrum = density_valid.mean(axis=0)
p10, p50, p90 = np.percentile(density_valid, [10, 50, 90], axis=0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(freqs, p10, p90, alpha=0.2, label="10th-90th percentile")
plot_spectrum(ax, p50, freqs, label="median", color="tab:blue")
plot_spectrum(ax, mean_spectrum, freqs, label="mean", color="tab:red", linestyle="--")
ax.legend()
ax.set_title(f"Buoy {BUOY_ID}: climatological spectrum")
plt.tight_layout()
plt.show()


# Peak Detection & Sea-State Modality

`utils.find_significant_peaks` applies the Portilla et al. (2009, section 2b.2) four-criterion
test for a *physically real* spectral peak to a single spectrum. The number of surviving peaks
directly gives the modality of the sea state: 0 (no significant peak / flat-calm), 1 (unimodal),
2 (bimodal), 3 (trimodal), 4+ (complex multi-system sea).

`find_significant_peaks` runs `scipy.signal.find_peaks` plus a small Python loop per call — on
the order of ~1ms per timestamp, so a full multi-year hourly record (tens of thousands of rows)
takes on the order of a minute. `STEP` subsamples the record for a quicker look; set it to `1`
for the full record.

In [ ]:
STEP = 1  # increase (e.g. 6, 24) to subsample for a faster, coarser look

def compute_peak_stats(density_df, freqs, step=1, **peak_kwargs):
    """Per-timestamp significant-peak count and (fp, S_at_fp) pairs.

    Returns a DataFrame indexed like density_df.iloc[::step] with columns
    'n_peaks' and 'peak_idxs' (list[int], possibly empty).
    """
    sub = density_df.iloc[::step]
    n_peaks = np.empty(len(sub), dtype=int)
    peak_idxs = [None] * len(sub)
    for i, spectrum in enumerate(sub.values):
        if spectrum.max() <= 0:
            n_peaks[i] = 0
            peak_idxs[i] = []
            continue
        peaks = find_significant_peaks(freqs, spectrum, **peak_kwargs)
        n_peaks[i] = len(peaks)
        peak_idxs[i] = peaks
    return pd.DataFrame({"n_peaks": n_peaks, "peak_idxs": peak_idxs}, index=sub.index)

peak_stats = compute_peak_stats(density, freqs, step=STEP)
peak_stats["n_peaks"].value_counts().sort_index()


In [ ]:
modality_labels = {0: "0 (calm)", 1: "1 (unimodal)", 2: "2 (bimodal)", 3: "3 (trimodal)"}
counts = peak_stats["n_peaks"].clip(upper=4).value_counts(normalize=True).sort_index() * 100
labels = [modality_labels.get(k, "4+ (complex)") for k in counts.index]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(labels, counts.values, color="tab:blue")
ax.set_ylabel("% of records")
ax.set_title(f"Buoy {BUOY_ID}: sea-state modality")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
monthly_modality = (
    peak_stats["n_peaks"].clip(upper=4)
    .groupby(peak_stats.index.month)
    .value_counts(normalize=True)
    .unstack(fill_value=0) * 100
)
monthly_modality.columns = [modality_labels.get(c, "4+ (complex)") for c in monthly_modality.columns]

fig, ax = plt.subplots(figsize=(10, 5))
monthly_modality.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")
ax.set_xlabel("Month")
ax.set_ylabel("% of records")
ax.set_title(f"Buoy {BUOY_ID}: sea-state modality by month")
ax.legend(title="Modality", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


# Wind-Sea vs Swell Classification

Each significant peak is labeled `'wind_sea'` or `'swell'` via `utils.classify_partition`
(γ* = S_obs(fp)/S_PM(fp) against the Pierson-Moskowitz reference, threshold 1.0, per
Violante-Carvalho 2009). A spectrum can carry both labels at once (a mixed sea: a local wind-sea
riding on a background swell) — this is itself a key climatological fact about a location.

In [ ]:
def label_row(row):
    if not row["peak_idxs"]:
        return []
    spectrum = density.loc[row.name].values
    parts = classify_partitions(freqs, spectrum, row["peak_idxs"])
    return [p["label"] for p in parts]

peak_stats["labels"] = peak_stats.apply(label_row, axis=1)
has_wind_sea = peak_stats["labels"].apply(lambda ls: "wind_sea" in ls)
has_swell = peak_stats["labels"].apply(lambda ls: "swell" in ls)

print(f"Wind-sea present:      {has_wind_sea.mean():.1%}")
print(f"Swell present:         {has_swell.mean():.1%}")
print(f"Mixed (both present):  {(has_wind_sea & has_swell).mean():.1%}")
print(f"Swell-only:            {(has_swell & ~has_wind_sea).mean():.1%}")
print(f"Wind-sea-only:         {(has_wind_sea & ~has_swell).mean():.1%}")
print(f"Neither (no peaks):    {(~has_wind_sea & ~has_swell).mean():.1%}")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
categories = ["Mixed\n(wind-sea + swell)", "Swell only", "Wind-sea only", "No significant peak"]
values = [
    (has_wind_sea & has_swell).mean(),
    (has_swell & ~has_wind_sea).mean(),
    (has_wind_sea & ~has_swell).mean(),
    (~has_wind_sea & ~has_swell).mean(),
]
ax.bar(categories, np.array(values) * 100, color=["tab:purple", "tab:cyan", "tab:orange", "tab:gray"])
ax.set_ylabel("% of records")
ax.set_title(f"Buoy {BUOY_ID}: sea-state composition")
plt.xticks(rotation=15)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


# Swell Evolution Case Study

Swell partitions are narrow-banded and persistent, and classically show a *frequency downshift*
(dispersion: longer-period components arrive/persist as the group ages) together with gradual
energy decay — unlike a wind-sea partition, which fluctuates faster and more broadly with local
wind forcing. Pick the longest run of consecutive timestamps carrying a swell peak in a similar
frequency band and track that partition's peak frequency and partition-local Hs across the
window.

In [ ]:
def swell_peak_freq(row):
    """Frequency of the lowest-frequency swell peak in a row, or NaN if none."""
    if not row["peak_idxs"]:
        return np.nan
    spectrum = density.loc[row.name].values
    parts = classify_partitions(freqs, spectrum, row["peak_idxs"])
    swell_fps = [p["fp"] for p in parts if p["label"] == "swell"]
    return min(swell_fps) if swell_fps else np.nan

swell_fp = peak_stats.apply(swell_peak_freq, axis=1)

# Longest run of consecutive timestamps with a swell peak in a stable frequency band.
has_swell_series = swell_fp.notna()
run_id = (has_swell_series != has_swell_series.shift()).cumsum()
run_lengths = has_swell_series.groupby(run_id).transform("size")
candidate_runs = run_lengths.where(has_swell_series)
best_run_id = run_id[candidate_runs == candidate_runs.max()].iloc[0]
case_window = swell_fp.index[run_id == best_run_id]

print(f"Case-study window: {case_window.min()} -> {case_window.max()} "
      f"({len(case_window)} hourly records)")


In [ ]:
def partition_hs(t, freqs_arr):
    spectrum = density.loc[t].values
    windows = find_peak_windows(freqs, spectrum)
    if not windows:
        return np.nan
    fp_target = swell_fp[t]
    # nearest surviving peak to the swell frequency identified above
    peak_idx, left, right = min(windows, key=lambda w: abs(freqs[w[0]] - fp_target))
    f_win = freqs[left:right + 1]
    return compute_hs(spectrum[left:right + 1][np.newaxis, :], f_win)[0]

case_fp = swell_fp.loc[case_window]
case_hs = pd.Series([partition_hs(t, freqs) for t in case_window], index=case_window)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(case_fp.index, case_fp.values, marker="o", ms=3)
axes[0].set_ylabel("Swell peak frequency [Hz]")
axes[0].grid(True)
axes[1].plot(case_hs.index, case_hs.values, marker="o", ms=3, color="tab:red")
axes[1].set_ylabel("Swell partition Hs [m]")
axes[1].set_xlabel("Time")
axes[1].grid(True)
plt.suptitle(f"Buoy {BUOY_ID}: swell evolution case study")
plt.tight_layout()
plt.show()


# Directional Structure

`alpha_1` is the per-frequency mean wave direction and `r_1`/`r_2` are the first/second
normalized directional Fourier coefficients (0-1). Standard formula (Kuik et al. 1988; the same
convention NDBC uses for its published directional parameters) for the directional spreading
width at a given frequency bin from `r_1`:

`sigma_1 = sqrt(2 * (1 - r_1))` radians

A narrow swell partition should show small `sigma_1`; a broad, locally-forced wind-sea partition
should show a larger one. Wind-sea direction should also track the concurrent wind direction
more closely than swell direction does (swell can arrive from a distant, unrelated storm).

Note: some buoys carry no working anemometer (`WDIR`/`WSPD` are all NDBC missing-value
sentinels, so `wind_u`/`wind_v` are all-NaN after `process_wind`) — the wind-alignment panel
below will be empty in that case; the spreading-by-partition-type panel is unaffected since it
doesn't depend on wind.

In [ ]:
def circular_mean_deg(degrees):
    rad = np.radians(np.asarray(degrees))
    return np.degrees(np.arctan2(np.sin(rad).mean(), np.cos(rad).mean())) % 360

def spreading_width_deg(r1):
    return np.degrees(np.sqrt(2 * np.clip(1 - r1, 0, None)))

def angular_diff_deg(a, b):
    """Smallest absolute difference between two angles in degrees, in [0, 180]."""
    d = np.abs(np.asarray(a) - np.asarray(b)) % 360
    return np.minimum(d, 360 - d)


In [ ]:
records = []
for t, row in peak_stats.iterrows():
    if not row["peak_idxs"]:
        continue
    spectrum = density.loc[t].values
    parts = classify_partitions(freqs, spectrum, row["peak_idxs"])
    for p, idx in zip(parts, row["peak_idxs"]):
        records.append({
            "time": t,
            "label": p["label"],
            "direction": alpha_1.loc[t].values[idx],
            "spreading": spreading_width_deg(r_1.loc[t].values[idx]),
        })

partitions_df = pd.DataFrame.from_records(records)
partitions_df["wind_dir"] = wind_dir.reindex(partitions_df["time"]).values
partitions_df["dir_wind_diff"] = angular_diff_deg(partitions_df["direction"], partitions_df["wind_dir"])
partitions_df.groupby("label")[["spreading", "dir_wind_diff"]].describe().T


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for label, color in [("wind_sea", "tab:orange"), ("swell", "tab:cyan")]:
    subset = partitions_df.loc[partitions_df["label"] == label, "spreading"].dropna()
    if len(subset):
        axes[0].hist(subset, bins=30, alpha=0.6, label=label, color=color)
axes[0].set_xlabel("Directional spreading σ₁ [deg]")
axes[0].set_ylabel("Count")
axes[0].set_title("Spreading by partition type")
axes[0].legend()
axes[0].grid(True)

for label, color in [("wind_sea", "tab:orange"), ("swell", "tab:cyan")]:
    subset = partitions_df.loc[partitions_df["label"] == label, "dir_wind_diff"].dropna()
    if len(subset):
        axes[1].hist(subset, bins=30, alpha=0.6, label=label, color=color)
    else:
        axes[1].text(0.5, 0.5, f"no wind data\n({label})", ha="center", va="center",
                      transform=axes[1].transAxes)
axes[1].set_xlabel("|Peak direction - wind direction| [deg]")
axes[1].set_title("Alignment with local wind")
if axes[1].get_legend_handles_labels()[0]:
    axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


# Summary

Fill in per-buoy observations here after running the notebook, e.g.:

- Sea-state modality: __% unimodal, __% bimodal, __% trimodal or more.
- Wind-sea present __% of the time, swell present __% of the time, mixed __%.
- Swell partitions show [narrower / wider] directional spreading than wind-sea partitions.
- Wind-sea direction tracks wind direction within roughly __° on average; swell does not.
- Notable seasonal pattern in modality: ...
